# imports

In [76]:
from pathlib import Path
import ast
import json
import time
import copy
from collections import Counter

import cv2
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import f1_score, classification_report

# raíz del proyecto y device

In [77]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "nih_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
OUTPUT_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/nih_baseline
DEVICE: cuda
GPU: NVIDIA GeForce RTX 4060


# cargar subset y label map

In [78]:
subset_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_large_with_paths.csv"
manifest_full_path = MANIFESTS_DIR / "manifest_nih_final_with_paths.csv"

subset_df = pd.read_csv(subset_path)
manifest_full = pd.read_csv(manifest_full_path)

print("subset_df:", subset_df.shape)
print("manifest_full:", manifest_full.shape)
print("\nConteo por split:")
print(subset_df["split_final"].value_counts())

subset_df: (19000, 19)
manifest_full: (112120, 19)

Conteo por split:
split_final
train    15000
val       2000
test      2000
Name: count, dtype: int64


# parsear labels

In [79]:
def safe_parse_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if not x:
        return []
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [label.strip() for label in x.split("|") if label.strip()]

def clean_labels(labels):
    return [label for label in labels if label != "No Finding"]

subset_df["labels_list"] = subset_df["labels_list"].apply(safe_parse_labels).apply(clean_labels)
manifest_full["labels_list"] = manifest_full["labels_list"].apply(safe_parse_labels).apply(clean_labels)

subset_df[["image_name", "labels_list"]].head()

,image_name,labels_list
0,00022245_021.png,[]
1,00019544_000.png,[]
2,00009673_001.png,[Pleural_Thickening]
3,00018103_001.png,[]
4,00017799_000.png,[Nodule]


# separar train / val / test

In [80]:
train_df = subset_df[subset_df["split_final"] == "train"].copy()
val_df = subset_df[subset_df["split_final"] == "val"].copy()
test_df = subset_df[subset_df["split_final"] == "test"].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (15000, 19)
val: (2000, 19)
test: (2000, 19)


In [81]:
# ===== sampler por rareza de patologías =====
train_label_counts = Counter()
for labels in train_df["labels_list"]:
    train_label_counts.update(labels)

train_label_counts = dict(train_label_counts)
print("Frecuencia de etiquetas en train:")
print(train_label_counts)

# peso inverso por patología
label_inv_freq = {
    label: 1.0 / count
    for label, count in train_label_counts.items()
}

# peso por imagen = promedio de inverse frequency de sus patologías
sample_weights = []
for labels in train_df["labels_list"]:
    if len(labels) == 0:
        w = 1.0
    else:
        w = float(np.mean([label_inv_freq[label] for label in labels]))
    sample_weights.append(w)

sample_weights = np.array(sample_weights, dtype=np.float32)

# estabilizar rango: normalizamos y capamos
sample_weights = sample_weights / sample_weights.mean()
sample_weights = np.clip(sample_weights, 0.5, 5.0)

sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.double)

print("\nResumen de sample_weights:")
print("min:", sample_weights.min())
print("mean:", sample_weights.mean())
print("max:", sample_weights.max())
print("percentiles:", np.percentile(sample_weights, [5, 25, 50, 75, 95, 99]))

Frecuencia de etiquetas en train:
{'Pleural_Thickening': 370, 'Nodule': 781, 'Fibrosis': 211, 'Effusion': 1493, 'Atelectasis': 1423, 'Mass': 681, 'Cardiomegaly': 311, 'Infiltration': 2354, 'Pneumothorax': 451, 'Hernia': 22, 'Consolidation': 488, 'Edema': 240, 'Emphysema': 223, 'Pneumonia': 150}

Resumen de sample_weights:
min: 0.5
mean: 1.2052898
max: 1.700834
percentiles: [0.5        0.5        1.70083404 1.70083404 1.70083404 1.70083404]


## prueba esta es la 5.5 celda


In [82]:
all_labels = sorted({
    label
    for labels in manifest_full["labels_list"]
    for label in labels
})

label_to_idx = {label: i for i, label in enumerate(all_labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

print("Etiquetas finales usadas:")
print(all_labels)
print("\nNúmero de etiquetas:", len(all_labels))

Etiquetas finales usadas:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

Número de etiquetas: 14


# función multihot

In [83]:
def labels_to_multihot(labels, label_to_idx):
    vec = torch.zeros(len(label_to_idx), dtype=torch.float32)
    for label in labels:
        if label in label_to_idx:
            vec[label_to_idx[label]] = 1.0
    return vec

## esta tambien es de prueba 6.5 celda

In [84]:
# ===== pos_weight suavizado =====
train_targets_matrix = np.stack(train_df["target"].values) if "target" in train_df.columns else None

if train_targets_matrix is None:
    # construir matriz multilabel desde labels_list
    train_targets_matrix = []
    for labels in train_df["labels_list"]:
        target = labels_to_multihot(labels, label_to_idx).numpy()
        train_targets_matrix.append(target)
    train_targets_matrix = np.stack(train_targets_matrix, axis=0)

pos_counts = train_targets_matrix.sum(axis=0)
neg_counts = len(train_targets_matrix) - pos_counts

raw_pos_weight = neg_counts / np.clip(pos_counts, a_min=1.0, a_max=None)

# suavizamos para no empujar tanto al modelo a sobrepredecir positivos
smoothed_pos_weight = np.sqrt(raw_pos_weight)

# además capamos el máximo
smoothed_pos_weight = np.clip(smoothed_pos_weight, 1.0, 5.0)

pos_weight = torch.tensor(smoothed_pos_weight, dtype=torch.float32)

print("raw_pos_weight:")
print(raw_pos_weight)

print("\nsmoothed_pos_weight:")
print(smoothed_pos_weight)

raw_pos_weight:
[  9.54111    47.23151    29.737705   61.5         9.0468855  66.26457
  70.09005   680.8182      5.372133   21.026432   18.206146   39.54054
  99.         32.259422 ]

smoothed_pos_weight:
[3.0888686 5.        5.        5.        3.0078042 5.        5.
 5.        2.3177862 4.5854588 4.266866  5.        5.        5.       ]


# transforms

In [85]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4

USE_CLAHE = True
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)

USE_MIXED_PRECISION = True

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def to_gray_uint8(path_str: str) -> np.ndarray:
    path = Path(path_str)

    with Image.open(path) as img:
        arr = np.array(img)

    if arr.ndim == 3:
        if arr.shape[2] == 4:
            arr = arr[:, :, :3]

        if arr.dtype != np.uint8:
            arr = arr.astype(np.float32)
            mn, mx = arr.min(), arr.max()
            if mx > mn:
                arr = ((arr - mn) / (mx - mn) * 255.0).clip(0, 255).astype(np.uint8)
            else:
                arr = np.zeros(arr.shape[:2] + (arr.shape[2],), dtype=np.uint8)

        arr = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

    elif arr.ndim == 2:
        if arr.dtype != np.uint8:
            arr = arr.astype(np.float32)
            mn, mx = arr.min(), arr.max()
            if mx > mn:
                arr = ((arr - mn) / (mx - mn) * 255.0).clip(0, 255).astype(np.uint8)
            else:
                arr = np.zeros_like(arr, dtype=np.uint8)
    else:
        raise ValueError(f"Formato no soportado: shape={arr.shape}")

    return arr


def apply_clahe(gray_uint8: np.ndarray,
                clip_limit: float = 2.0,
                tile_grid_size=(8, 8)) -> np.ndarray:
    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )
    return clahe.apply(gray_uint8)


def preprocess_nih_image(path_str: str,
                         image_size: int = 224,
                         use_clahe: bool = True,
                         clahe_clip_limit: float = 2.0,
                         clahe_tile_grid_size=(8, 8)) -> Image.Image:
    gray = to_gray_uint8(path_str)

    if use_clahe:
        gray = apply_clahe(
            gray,
            clip_limit=clahe_clip_limit,
            tile_grid_size=clahe_tile_grid_size
        )

    pil_gray = Image.fromarray(gray)
    pil_gray = pil_gray.resize((image_size, image_size), Image.BILINEAR)

    # 3 canales para backbone preentrenado
    pil_rgb = Image.merge("RGB", (pil_gray, pil_gray, pil_gray))
    return pil_rgb


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.RandomAffine(degrees=0, translate=(0.02, 0.02)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
])

print("IMAGE_SIZE:", IMAGE_SIZE)
print("USE_CLAHE:", USE_CLAHE)
print("USE_MIXED_PRECISION:", USE_MIXED_PRECISION)

IMAGE_SIZE: 224
USE_CLAHE: True
USE_MIXED_PRECISION: True


# dataset

In [86]:
class NIHSubsetDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["file_path"])
        image = preprocess_nih_image(
            path_str=str(image_path),
            image_size=IMAGE_SIZE,
            use_clahe=USE_CLAHE,
            clahe_clip_limit=CLAHE_CLIP_LIMIT,
            clahe_tile_grid_size=CLAHE_TILE_GRID_SIZE
        )

        if self.transform is not None:
            image = self.transform(image)

        target = labels_to_multihot(row["labels_list"], self.label_to_idx)

        return {
            "image": image,
            "target": target,
            "image_name": row["image_name"],
            "labels_text": " | ".join(row["labels_list"]),
        }

# datasets y dataloaders

In [87]:
train_dataset = NIHSubsetDataset(train_df, label_to_idx, transform=train_transform)
val_dataset = NIHSubsetDataset(val_df, label_to_idx, transform=eval_transform)
test_dataset = NIHSubsetDataset(test_df, label_to_idx, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

print("Datasets y dataloaders listos.")
print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

Datasets y dataloaders listos.
train batches: 469
val batches: 63
test batches: 63


# modelo baseline

In [88]:
NUM_CLASSES = len(label_to_idx)

model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
model = model.to(DEVICE)

print(model.classifier)
print("Número de clases:", NUM_CLASSES)

Linear(in_features=1024, out_features=14, bias=True)
Número de clases: 14


In [89]:
def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

freeze_backbone(model)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Parámetros entrenables:", trainable_params)
print("Parámetros totales:", total_params)

Parámetros entrenables: 14350
Parámetros totales: 6968206


# loss, optimizer y scheduler

In [90]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda" and USE_MIXED_PRECISION))

/tmp/ipykernel_1523/3787093755.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda" and USE_MIXED_PRECISION))


In [91]:
def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

unfreeze_lr = 2e-4

# función de evaluación

In [92]:
def apply_thresholds(probs: np.ndarray, thresholds):
    """
    probs: [N, C]
    thresholds: escalar o array/lista de shape [C]
    """
    thresholds = np.array(thresholds, dtype=np.float32)
    if thresholds.ndim == 0:
        thresholds = np.full((probs.shape[1],), float(thresholds), dtype=np.float32)

    preds = (probs >= thresholds.reshape(1, -1)).astype(int)
    return preds

In [93]:
def evaluate_model(model, loader, criterion, device, thresholds=0.3):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device == "cuda" and USE_MIXED_PRECISION)):
                logits = model(images)
                loss = criterion(logits, targets)

            probs = torch.sigmoid(logits)

            total_loss += loss.item() * images.size(0)
            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = apply_thresholds(all_probs, thresholds)

    f1_macro = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    f1_micro = f1_score(all_targets, all_preds, average="micro", zero_division=0)

    return avg_loss, f1_macro, f1_micro

# loop de entrenamiento

In [94]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device == "cuda" and USE_MIXED_PRECISION)):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss

# entrenamiento completo

In [96]:
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-4

history = []
best_val_f1_macro = -np.inf
best_model_state = copy.deepcopy(model.state_dict())
best_epoch = 0
epochs_without_improvement = 0

PHASE1_EPOCHS = 4
PHASE2_EPOCHS = 10
current_epoch = 0

print("=== FASE 1: entrenamiento de la cabeza ===")

for local_epoch in range(1, PHASE1_EPOCHS + 1):
    current_epoch += 1
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_f1_macro, val_f1_micro = evaluate_model(
        model, val_loader, criterion, DEVICE, thresholds=0.3
    )

    scheduler.step(val_f1_macro)
    elapsed = time.time() - start_time

    history.append({
        "epoch": current_epoch,
        "phase": 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1_macro,
        "val_f1_micro": val_f1_micro,
        "lr": optimizer.param_groups[0]["lr"],
        "time_sec": elapsed,
    })

    print(
        f"[F1] Epoch {local_epoch:02d}/{PHASE1_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f} | "
        f"val_f1_micro={val_f1_micro:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.6f} | "
        f"time={elapsed:.1f}s"
    )

    if val_f1_macro > (best_val_f1_macro + EARLY_STOPPING_MIN_DELTA):
        best_val_f1_macro = val_f1_macro
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = current_epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping activado en fase 1, época global {current_epoch}.")
        break

print("\n=== FASE 2: fine-tuning completo ===")

unfreeze_all(model)

optimizer = optim.AdamW(model.parameters(), lr=unfreeze_lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

for local_epoch in range(1, PHASE2_EPOCHS + 1):
    current_epoch += 1
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_f1_macro, val_f1_micro = evaluate_model(model, val_loader, criterion, DEVICE, thresholds=0.3)

    scheduler.step(val_f1_macro)
    elapsed = time.time() - start_time

    history.append({
        "epoch": current_epoch,
        "phase": 2,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1_macro,
        "val_f1_micro": val_f1_micro,
        "lr": optimizer.param_groups[0]["lr"],
        "time_sec": elapsed,
    })

    print(
        f"[F2] Epoch {local_epoch:02d}/{PHASE2_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f} | "
        f"val_f1_micro={val_f1_micro:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.6f} | "
        f"time={elapsed:.1f}s"
    )

    if val_f1_macro > (best_val_f1_macro + EARLY_STOPPING_MIN_DELTA):
        best_val_f1_macro = val_f1_macro
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = current_epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping activado en fase 2, época global {current_epoch}.")
        break

print(f"\nEntrenamiento terminado. Mejor época global: {best_epoch} | best_val_f1_macro={best_val_f1_macro:.4f}")

=== FASE 1: entrenamiento de la cabeza ===
[F1] Epoch 01/4 | train_loss=0.4126 | val_loss=0.4263 | val_f1_macro=0.1065 | val_f1_micro=0.2346 | lr=0.001000 | time=148.2s
[F1] Epoch 02/4 | train_loss=0.3922 | val_loss=0.4298 | val_f1_macro=0.1052 | val_f1_micro=0.2120 | lr=0.001000 | time=148.4s
[F1] Epoch 03/4 | train_loss=0.3859 | val_loss=0.4208 | val_f1_macro=0.1378 | val_f1_micro=0.2426 | lr=0.001000 | time=147.6s
[F1] Epoch 04/4 | train_loss=0.3831 | val_loss=0.4250 | val_f1_macro=0.1236 | val_f1_micro=0.2575 | lr=0.001000 | time=147.6s

=== FASE 2: fine-tuning completo ===
[F2] Epoch 01/10 | train_loss=0.3727 | val_loss=0.3897 | val_f1_macro=0.1933 | val_f1_micro=0.2945 | lr=0.000200 | time=153.7s
[F2] Epoch 02/10 | train_loss=0.3468 | val_loss=0.3817 | val_f1_macro=0.1884 | val_f1_micro=0.2835 | lr=0.000200 | time=157.6s
[F2] Epoch 03/10 | train_loss=0.3315 | val_loss=0.3762 | val_f1_macro=0.2165 | val_f1_micro=0.3121 | lr=0.000200 | time=153.3s
[F2] Epoch 04/10 | train_loss=0.32

In [ ]:
def collect_probs_targets(model, loader, device):
    model.eval()
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device == "cuda" and USE_MIXED_PRECISION)):
                logits = model(images)

            probs = torch.sigmoid(logits)

            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    return all_targets, all_probs

# cargar mejor modelo y evaluar en test

In [ ]:
model.load_state_dict(best_model_state)

test_targets, test_probs = collect_probs_targets(model, test_loader, DEVICE)
test_preds = apply_thresholds(test_probs, best_thresholds_per_class)

test_loss, _, _ = evaluate_model(
    model,
    test_loader,
    criterion,
    DEVICE,
    thresholds=best_thresholds_per_class
)

test_f1_macro = f1_score(test_targets, test_preds, average="macro", zero_division=0)
test_f1_micro = f1_score(test_targets, test_preds, average="micro", zero_division=0)

print("Resultados en test:")
print("thresholds por clase =", best_thresholds_per_class)
print(f"test_loss       = {test_loss:.4f}")
print(f"test_f1_macro   = {test_f1_macro:.4f}")
print(f"test_f1_micro   = {test_f1_micro:.4f}")

print("\nClassification report:")
print(
    classification_report(
        test_targets,
        test_preds,
        target_names=[idx_to_label[i] for i in range(len(idx_to_label))],
        zero_division=0
    )
)

Resultados en test:
thresholds por clase = [0.1  0.1  0.1  0.1  0.15 0.2  0.1  0.1  0.1  0.25 0.1  0.1  0.1  0.1 ]
test_loss       = 0.2413
test_f1_macro   = 0.2156
test_f1_micro   = 0.3347

Classification report:
                    precision    recall  f1-score   support

       Atelectasis       0.25      0.55      0.34       258
      Cardiomegaly       0.46      0.14      0.21        93
     Consolidation       0.15      0.29      0.20       148
             Edema       0.13      0.33      0.18        66
          Effusion       0.42      0.54      0.47       386
         Emphysema       0.22      0.10      0.14        88
          Fibrosis       0.00      0.00      0.00        42
            Hernia       0.00      0.00      0.00         2
      Infiltration       0.33      0.64      0.44       459
              Mass       0.28      0.27      0.27       150
            Nodule       0.20      0.26      0.23       129
Pleural_Thickening       0.09      0.18      0.12        92
     

# guardar historial y checkpoint

In [ ]:
history_df = pd.DataFrame(history)
history_path = OUTPUT_DIR / "nih_baseline_history.csv"
history_df.to_csv(history_path, index=False)

checkpoint_path = OUTPUT_DIR / "nih_baseline_densenet121_subset_large.pth"
torch.save({
    "model_state_dict": best_model_state,
    "label_to_idx": label_to_idx,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "best_val_f1_macro": best_val_f1_macro,
    "best_epoch": best_epoch,
    "best_thresholds_per_class": best_thresholds_per_class.tolist(),
    "test_loss": float(test_loss),
    "test_f1_macro": float(test_f1_macro),
    "test_f1_micro": float(test_f1_micro),
}, checkpoint_path)

print("Historial guardado en:", history_path)
print("Checkpoint guardado en:", checkpoint_path)

Historial guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/nih_baseline/nih_baseline_history.csv
Checkpoint guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/nih_baseline/nih_baseline_densenet121_subset_large.pth


In [ ]:
model.load_state_dict(best_model_state)

test_targets, test_probs = collect_probs_targets(model, test_loader, DEVICE)
test_preds = apply_thresholds(test_probs, best_thresholds_per_class)

test_loss, _, _ = evaluate_model(
    model,
    test_loader,
    criterion,
    DEVICE,
    thresholds=best_thresholds_per_class
)

test_f1_macro = f1_score(test_targets, test_preds, average="macro", zero_division=0)
test_f1_micro = f1_score(test_targets, test_preds, average="micro", zero_division=0)

print("Resultados en test:")
print("thresholds por clase =", best_thresholds_per_class)
print(f"test_loss       = {test_loss:.4f}")
print(f"test_f1_macro   = {test_f1_macro:.4f}")
print(f"test_f1_micro   = {test_f1_micro:.4f}")

print("\nClassification report:")
print(
    classification_report(
        test_targets,
        test_preds,
        target_names=[idx_to_label[i] for i in range(len(idx_to_label))],
        zero_division=0
    )
)

,class_idx,label_name,best_threshold,best_binary_f1
0,0,Atelectasis,0.10,0.323843
1,1,Cardiomegaly,0.10,0.117647
2,2,Consolidation,0.10,0.075000
3,3,Edema,0.10,0.259740
4,4,Effusion,0.15,0.489978
5,5,Emphysema,0.20,0.153846
6,6,Fibrosis,0.10,0.000000
7,7,Hernia,0.10,0.000000
8,8,Infiltration,0.10,0.332362
9,9,Mass,0.25,0.257669


Thresholds por clase:
[0.1  0.1  0.1  0.1  0.15 0.2  0.1  0.1  0.1  0.25 0.1  0.1  0.1  0.1 ]

Val F1 macro con thresholds por clase: 0.1825
Val F1 micro con thresholds por clase: 0.2942


In [ ]:
history_df

,epoch,phase,train_loss,val_loss,val_f1_macro,val_f1_micro,lr,time_sec
0,1,1,1.350072,1.328496,0.098476,0.101491,0.0010,342.486326
1,2,1,1.219183,1.335070,0.102766,0.112360,0.0010,160.496561
2,3,1,1.188083,1.326710,0.102533,0.111996,0.0010,166.099800
3,4,1,1.167124,1.379470,0.112264,0.129404,0.0010,163.386203
4,5,2,1.182893,1.374367,0.111028,0.117271,0.0002,175.200128
5,6,2,1.110485,1.258204,0.117980,0.132345,0.0002,173.770525
6,7,2,1.046925,1.179763,0.125875,0.141166,0.0002,164.069487
7,8,2,1.008910,1.201454,0.117233,0.133441,0.0002,171.942282
8,9,2,0.970756,1.273900,0.137943,0.162896,0.0002,175.260597
9,10,2,0.942295,1.262619,0.140711,0.159161,0.0002,169.769110
